In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install timesfm==2.0.2

In [ ]:
import sys,os
sys.path.append('/content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting/src')

In [ ]:
import numpy as np,pandas as pd
import config as cf, data, predictions

In [ ]:
#chck and build the daily model input file
if os.path.exists(cf.MODEL_INPUT):
    df = data.load()
else:
    df = data.build()

print(len(df), 'rows')

4850000 rows


In [ ]:
# load the model
import timesfm

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch",
    torch_compile=False)

model.compile(timesfm.ForecastConfig(
    max_context=1024,
    max_horizon=256,
    normalize_inputs=True,
    use_continuous_quantile_head=True,
    force_flip_invariance=True,
    infer_is_positive=True,
    fix_quantile_crossing=True,
    return_backcast=True))

In [ ]:
# features

categorical=['holiday_flag', 'activity_flag']
numerical=[c for c in cf.EXOG if c not in categorical]

assert len(categorical) + len(numerical) == len(cf.EXOG) , 'all features not passed'

In [ ]:
# forecasts every series and saves one file
def run_timesfm(train_df, predict_df, input_version, split, batch=1000):

    pred_dates = [pd.Timestamp(d).strftime('%Y-%m-%d')
                  for d in sorted(pd.unique(predict_df['ds']))[:cf.HORIZON]]
    train_df = train_df.sort_values(['unique_id', 'ds'])
    predict_df = predict_df.sort_values(['unique_id', 'ds'])
    past_of = dict(tuple(train_df.groupby('unique_id', sort=False)))
    future_of = dict(tuple(predict_df.groupby('unique_id', sort=False)))

    series_ids = train_df['unique_id'].unique().tolist()
    done_ids, done_preds = [], []
    for start in range(0, len(series_ids), batch):
        chunk = series_ids[start:start + batch]

        history = []
        numbers = {c: [] for c in numerical}
        flags = {c: [] for c in categorical}
        statics = {c: [] for c in cf.STATIC}

        for uid in chunk:
            past, future = past_of[uid], future_of[uid]

            history.append(past['y'].to_numpy(np.float32))
            for c in numerical:
                numbers[c].append(np.concatenate([past[c].to_numpy(np.float32),
                                                  future[c].to_numpy(np.float32)]))
            for c in categorical:
                flags[c].append(np.concatenate([past[c].to_numpy(),
                                                future[c].to_numpy()]).astype(int).tolist())
            for c in cf.STATIC:
                statics[c].append(int(past[c].iloc[0]))

        forecast, _ = model.forecast_with_covariates(
            inputs=history,
            dynamic_numerical_covariates=numbers,
            dynamic_categorical_covariates=flags,
            static_categorical_covariates=statics,
            xreg_mode="xreg + timesfm",
            normalize_xreg_target_per_input=True)

        for uid, values in zip(chunk, forecast):
            done_ids.append(uid)
            done_preds.append(np.asarray(values).reshape(-1)[:cf.HORIZON])

        print(f'  {min(start + batch, len(series_ids))}/{len(series_ids)}')

    out = pd.DataFrame({
        'unique_id': np.repeat(done_ids, cf.HORIZON),
        'dt': np.tile(pred_dates, len(done_ids)),
        'prediction': np.concatenate(done_preds)})

    out[['store_id', 'product_id']] = out['unique_id'].str.split('_', expand=True).astype(int)

    out.to_parquet(f'/content/{input_version}_{split}_timesfm_backup.parquet', index=False)

    predictions.save(out[['store_id', 'product_id', 'dt', 'prediction']],
                     'TimesFM', input_version, split)

In [ ]:
!nvidia-smi

Wed Aug  5 14:35:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             62W /  400W |    1370MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
#  real sales, validation week. Check this works before the rest.
train_df, predict_df = data.splittbl(df, 'raw', 'val')
run_timesfm(train_df, predict_df, 'raw', 'val')

  1000/50000
  2000/50000
  3000/50000
  4000/50000
  5000/50000
  6000/50000
  7000/50000
  8000/50000
  9000/50000
  10000/50000
  11000/50000
  12000/50000
  13000/50000
  14000/50000
  15000/50000
  16000/50000
  17000/50000
  18000/50000
  19000/50000
  20000/50000
  21000/50000
  22000/50000
  23000/50000
  24000/50000
  25000/50000
  26000/50000
  27000/50000
  28000/50000
  29000/50000
  30000/50000
  31000/50000
  32000/50000
  33000/50000
  34000/50000
  35000/50000
  36000/50000
  37000/50000
  38000/50000
  39000/50000
  40000/50000
  41000/50000
  42000/50000
  43000/50000
  44000/50000
  45000/50000
  46000/50000
  47000/50000
  48000/50000
  49000/50000
  50000/50000
[val] 350,000 rows : TimesFM__raw


In [ ]:
# recovered demand, validation week
train_df, predict_df = data.splittbl(df, 'recovered', 'val')
run_timesfm(train_df, predict_df, 'recovered', 'val')

  1000/50000
  2000/50000
  3000/50000
  4000/50000
  5000/50000
  6000/50000
  7000/50000
  8000/50000
  9000/50000
  10000/50000
  11000/50000
  12000/50000
  13000/50000
  14000/50000
  15000/50000
  16000/50000
  17000/50000
  18000/50000
  19000/50000
  20000/50000
  21000/50000
  22000/50000
  23000/50000
  24000/50000
  25000/50000
  26000/50000
  27000/50000
  28000/50000
  29000/50000
  30000/50000
  31000/50000
  32000/50000
  33000/50000
  34000/50000
  35000/50000
  36000/50000
  37000/50000
  38000/50000
  39000/50000
  40000/50000
  41000/50000
  42000/50000
  43000/50000
  44000/50000
  45000/50000
  46000/50000
  47000/50000
  48000/50000
  49000/50000
  50000/50000
[val] 350,000 rows : TimesFM__recovered


In [ ]:
# real sales, test week
train_df, predict_df = data.splittbl(df, 'raw', 'test')
run_timesfm(train_df, predict_df, 'raw', 'test')

  1000/50000
  2000/50000
  3000/50000
  4000/50000
  5000/50000
  6000/50000
  7000/50000
  8000/50000
  9000/50000
  10000/50000
  11000/50000
  12000/50000
  13000/50000
  14000/50000
  15000/50000
  16000/50000
  17000/50000
  18000/50000
  19000/50000
  20000/50000
  21000/50000
  22000/50000
  23000/50000
  24000/50000
  25000/50000
  26000/50000
  27000/50000
  28000/50000
  29000/50000
  30000/50000
  31000/50000
  32000/50000
  33000/50000
  34000/50000
  35000/50000
  36000/50000
  37000/50000
  38000/50000
  39000/50000
  40000/50000
  41000/50000
  42000/50000
  43000/50000
  44000/50000
  45000/50000
  46000/50000
  47000/50000
  48000/50000
  49000/50000
  50000/50000
[test] 350,000 rows : TimesFM__raw


In [ ]:
# recovered demand, test week
train_df, predict_df = data.splittbl(df, 'recovered', 'test')
run_timesfm(train_df, predict_df, 'recovered', 'test')

  1000/50000
  2000/50000
  3000/50000
  4000/50000
  5000/50000
  6000/50000
  7000/50000
  8000/50000
  9000/50000
  10000/50000
  11000/50000
  12000/50000
  13000/50000
  14000/50000
  15000/50000
  16000/50000
  17000/50000
  18000/50000
  19000/50000
  20000/50000
  21000/50000
  22000/50000
  23000/50000
  24000/50000
  25000/50000
  26000/50000
  27000/50000
  28000/50000
  29000/50000
  30000/50000
  31000/50000
  32000/50000
  33000/50000
  34000/50000
  35000/50000
  36000/50000
  37000/50000
  38000/50000
  39000/50000
  40000/50000
  41000/50000
  42000/50000
  43000/50000
  44000/50000
  45000/50000
  46000/50000
  47000/50000
  48000/50000
  49000/50000
  50000/50000
[test] 350,000 rows : TimesFM__recovered


In [ ]:
# environment record
import platform, subprocess, sys
from importlib.metadata import version, PackageNotFoundError

print("OS      :", platform.platform())
print("Python  :", sys.version.split()[0])

try:
    import torch
    print("torch   :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU     :", torch.cuda.get_device_name(0),
              "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
except ImportError:
    print("torch   : not installed")

print("CPU     :", subprocess.run("nproc", capture_output=True, text=True).stdout.strip(), "cores")
print("RAM     :", subprocess.run("free -g | awk 'NR==2{print $2}'", shell=True,
                                  capture_output=True, text=True).stdout.strip(), "GB")

PKGS = ["neuralforecast", "mlforecast", "lightgbm", "statsforecast", "optuna",
        "timesfm", "chronos-forecasting", "pypots", "pandas", "numpy", "pyarrow",
        "scikit-learn", "pytorch-lightning"]
print()
for p in PKGS:
    try:
        print(f"{p:22s} {version(p)}")
    except PackageNotFoundError:
        print(f"{p:22s} -")

OS      : Linux-6.6.122+-x86_64-with-glibc2.35
Python  : 3.12.13
torch   : 2.11.0+cu128 | CUDA available: True
GPU     : NVIDIA A100-SXM4-80GB | 85.1 GB
CPU     : 12 cores
RAM     : 167 GB

neuralforecast         -
mlforecast             -
lightgbm               4.6.0
statsforecast          -
optuna                 -
timesfm                -
chronos-forecasting    -
pypots                 -
pandas                 2.2.2
numpy                  2.0.2
pyarrow                18.1.0
scikit-learn           1.6.1
pytorch-lightning      -
